# Install dependencies

In [ ]:
"""
QLoRA Finetuning — Qwen2.5-7B-Instruct
Dataset: RAG SFT dataset (ChatML format, ~700 samples)
Platform: Google Colab (A100 40GB / L4 22GB)
Framework: Unsloth + TRL SFTTrainer

Setup (run in Colab cell first):
    !pip install unsloth
    !pip install trl datasets
"""

# ── 0. Imports ────────────────────────────────────────────────────────────────
import json
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Hugging Face Login
Đăng nhập vào Hugging Face để tải dataset và đẩy model lên.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Config

In [ ]:
MODEL_NAME      = "unsloth/Qwen2.5-7B-Instruct"
DATASET_PATH    = "xunnhi/QA-Dataset-Generator"          # path tới file dataset của bạn
OUTPUT_DIR      = "./qwen_qlora_output"
MAX_SEQ_LENGTH  = 2048                     # Đã giảm xuống 2048 để an toàn cho T4 (RAG chunk của mình dài ~700 tokens nên 2048 là dư sức)

# QLoRA hyperparams
LORA_R          = 16     # rank
LORA_ALPHA      = 32     
LORA_DROPOUT    = 0.05
TARGET_MODULES  = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training hyperparams — Tuned cho 700 samples trên GPU T4 (15GB VRAM)
BATCH_SIZE          = 1    # Đã giảm từ 2 xuống 1 để chống tràn RAM GPU (OOM) trên T4
GRAD_ACCUM          = 16   # Tăng từ 8 lên 16 để giữ nguyên effective batch size (1 * 16 = 16)
NUM_EPOCHS          = 3
LEARNING_RATE       = 2e-4
WARMUP_RATIO        = 0.05
LR_SCHEDULER        = "cosine"
MAX_GRAD_NORM       = 1.0
WEIGHT_DECAY        = 0.01

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # auto-detect: bf16 on A100, fp16 on L4
    load_in_4bit=True,    # QLoRA
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # giảm VRAM thêm ~30%
    random_state=42,
)

print(model.print_trainable_parameters())

# Load + format dataset

In [ ]:
from datasets import load_dataset

def format_chatml(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


In [ ]:
# Load từ Hugging Face
raw_dataset = load_dataset(DATASET_PATH, split="train")

# Map qua hàm format_chatml
dataset = raw_dataset.map(format_chatml)
print(f"Dataset loaded: {len(dataset)} samples")


## Splitting

In [ ]:
# Train/eval split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

# Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        # Output
        output_dir=OUTPUT_DIR,

        # Training loop
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,

        # Optimizer
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        max_grad_norm=MAX_GRAD_NORM,
        optim="adamw_8bit",       # Unsloth 8-bit Adam — giảm VRAM optimizer state

        # Sequence
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=True,             # ghép các sequence ngắn vào cùng 1 batch — tăng throughput

        # Precision
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),

        # Logging & eval
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # Misc
        seed=42,
        report_to="none",         # đổi thành "wandb" nếu muốn track
        dataloader_num_workers=2,
    ),
)

# Training

In [ ]:
print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining done | {trainer_stats.metrics}")

# Report & Evaluation: Loss Curve
Vẽ biểu đồ Training Loss và Validation Loss để đánh giá xem mô hình có bị Overfitting hay không.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history

train_loss = [h["loss"] for h in history if "loss" in h]
train_steps = [h["step"] for h in history if "loss" in h]

eval_loss = [h["eval_loss"] for h in history if "eval_loss" in h]
eval_steps = [h["step"] for h in history if "eval_loss" in h]

plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_loss, label="Training Loss", alpha=0.8)
plt.plot(eval_steps, eval_loss, label="Validation Loss", color="orange", linewidth=2, marker="o")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training & Validation Loss Curve")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


# Save LoRA adapter + tokenizer

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved → {ADAPTER_DIR}")

# Optional: merge adapter vào base model để dùng như standalone model
# Bỏ comment nếu muốn export model merged (cần thêm ~14GB disk)
#
# MERGED_DIR = f"{OUTPUT_DIR}/merged_model"
# model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
# print(f"Merged model saved → {MERGED_DIR}")

# Quick inference test

In [ ]:
def test_inference(context: str, question: str) -> str:
    FastLanguageModel.for_inference(model)
    messages = [
        {"role": "system",    "content": "You are an expert scientific assistant. Answer questions based strictly on the provided context. If the context does not contain sufficient information, respond with INSUFFICIENT_INFORMATION."},
        {"role": "user",      "content": f"[Context]\n{context}\n\n[Question]\n{question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# Test với sample đơn giản
test_ctx = """
The proposed model uses a two-stage pipeline. In the first stage, document chunks are encoded
using a transformer backbone. In the second stage, a cross-attention mechanism aggregates
information across chunks to produce a final answer.
"""
test_q = "What are the two stages of the proposed pipeline?"

print("\n── Inference test ────────────────────────────────")
print(f"Q: {test_q}")
print(f"A: {test_inference(test_ctx, test_q)}")

# Push Model to Hugging Face Hub
Sau khi train xong, bạn có thể đẩy thẳng LoRA Adapter lên tài khoản Hugging Face của mình. Nhớ đổi tên repo thành tên bạn muốn nhé.

In [ ]:
HF_REPO = "xunnhi/Qwen2.5-7B-RAG-LoRA"  # Thay tên này thành repo bạn muốn tạo trên HF

# Đẩy LoRA adapter lên Hugging Face (cần chạy hf auth login từ trước, hoặc token được lưu trong Colab Secrets)
model.push_to_hub(HF_REPO, token=True)
tokenizer.push_to_hub(HF_REPO, token=True)

print(f"✅ Đã upload thành công LoRA Adapter lên: https://huggingface.co/{HF_REPO}")